# Short-Signal Demo: AAPL

This notebook walks through the full pipeline on a single ticker:

1. Download OHLCV data.
2. Build features (original 6 indicators + short-relevant additions).
3. Generate triple-barrier short labels.
4. Run purged walk-forward cross-validation.
5. Simulate the resulting signals with a simple backtest.
6. Inspect feature importances.

**Reminder:** this is a research baseline. It does not include borrow fees, slippage, or financing costs, and past performance on a single ticker is not evidence of an edge.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_prices
from src.features import build_features
from src.labels import BarrierConfig, triple_barrier_labels
from src.model import cross_validate, summarize_folds, fit_final
from src.backtest import backtest_signals, trade_stats, equity_curve

## 1. Download data

In [ ]:
TICKER = 'AAPL'
prices = load_prices(TICKER, start='2012-01-01')
print(prices.shape)
prices.tail()

## 2. Features and labels

The barrier config defines what counts as a 'good short':
- price drops 5% (`lower_pct`) before rising 3% (`upper_pct`)
- within 10 trading days (`horizon`)

Tweak these to see how the label distribution changes.

In [ ]:
cfg = BarrierConfig(lower_pct=0.05, upper_pct=0.03, horizon=10)

X = build_features(prices)
y_df = triple_barrier_labels(prices['Close'], prices['High'], prices['Low'], cfg)
common = X.index.intersection(y_df.index)
X = X.loc[common]
y = y_df.loc[common, 'label']

print(f'Feature matrix: {X.shape}')
print(f'Positive (good-short) rate: {y.mean():.3f}')
y_df['outcome'].value_counts()

## 3. Cross-validation

Purged walk-forward, 5 folds. The purge gap (10 days) prevents training labels from peeking at the test window.

In [ ]:
results = cross_validate(X, y, model_kind='rf', n_splits=5, purge=cfg.horizon)
summary = summarize_folds(results)
summary

**How to read this:** baseline precision is just `pos_rate_test` (the rate you'd get by always predicting positive). If model precision sits at or below the positive rate, the model has no edge on this ticker at this threshold.

In [ ]:
# Try the gradient-boosting alternative
hgb_results = cross_validate(X, y, model_kind='hgb', n_splits=5, purge=cfg.horizon)
summarize_folds(hgb_results)

## 4. Backtest the out-of-sample signals

In [ ]:
THRESHOLD = 0.55  # raise to be more selective (precision up, recall down)

sig_dates = []
for r in results:
    sig_dates.extend(r.test_index[r.y_proba >= THRESHOLD])
sig_dates = sorted(set(sig_dates))
print(f'{len(sig_dates)} signals at threshold {THRESHOLD}')

trades = backtest_signals(prices, sig_dates, cfg)
trade_stats(trades)

In [ ]:
eq = equity_curve(trades)
if len(eq):
    fig, ax = plt.subplots(figsize=(10, 4))
    eq.plot(ax=ax)
    ax.set_title(f'{TICKER} short-signal cumulative return (non-compounding, no costs)')
    ax.set_ylabel('Cumulative return')
    ax.axhline(0, color='black', linewidth=0.5)
    plt.show()

## 5. Feature importance

Fit a final model on all labeled data and inspect which features the RF leans on. Useful as a sanity check, not as causal evidence.

In [ ]:
final_model = fit_final(X, y, model_kind='rf')
importances = pd.Series(final_model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', figsize=(8, 5),
                                title=f'{TICKER} RF feature importance')
plt.tight_layout(); plt.show()

## 6. Score today

What signal would fire on the most recent bar?

In [ ]:
latest = X.iloc[[-1]]
proba = final_model.predict_proba(latest.to_numpy())[0, 1]
print(f'As of {latest.index[0].date()}: P(good short) = {proba:.3f}')
print('SIGNAL' if proba >= THRESHOLD else 'no signal')